In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/products.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/sample_submission.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/promotions.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/shipments.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/order_items.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/reviews.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/inventory.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/returns.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/sales.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/orders.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/geography.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/customers.csv
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/baseline.ipynb
/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/payments.csv
/kaggle/input/datasets/uynttrnnh/da

In [2]:
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np

In [3]:
# Đọc dữ liệu từ file csv
customers = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/customers.csv")
geography = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/geography.csv")
inventory = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/inventory.csv")
order_items = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/order_items.csv")
orders = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/orders.csv")
payments = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/payments.csv")
products = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/products.csv")
promotions = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/promotions.csv")
returns = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/returns.csv")
reviews = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/reviews.csv")
sales = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/sales.csv")
shipments = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/shipments.csv")
web_traffic = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/web_traffic.csv")

/tmp/ipykernel_55/1316847536.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv("/kaggle/input/datasets/uynttrnnh/datathon-2026-round-1/order_items.csv")


# **Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)**

> **Q1: C. 144 ngày**

In [4]:
# Chuẩn hoá cột order_date
orders["order_date"] = pd.to_datetime(orders["order_date"])

cust_cnt = orders.groupby('customer_id')['order_id'].count().reset_index()

# Khách mua >1 lần
multi_buyers = cust_cnt[cust_cnt['order_id'] > 1]

# Lọc orders chỉ giữ các khách này
tmp = orders.merge(
    multi_buyers[['customer_id']],
    on='customer_id',
    how='inner'
)

# Sort theo khách + ngày
tmp = tmp.sort_values(['customer_id', 'order_date'])

# Đơn trước đó
tmp['prev_order_date'] = tmp.groupby('customer_id')['order_date'].shift(1)

# Khoảng cách ngày
tmp['gap_days'] = (
    tmp['order_date'] - tmp['prev_order_date']
).dt.days

# Median
answer = tmp['gap_days'].dropna().median()

print(answer)

144.0


# **Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?**

> **Q2: D. Standard**

In [5]:
# Tạo cột gross margin
products['gross_margin'] = (
    products['price'] - products['cogs']
) / products['price']

# Gộp và tính trung bình theo segment
result = (
    products.groupby('segment')['gross_margin']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
# Tìm segment cao nhất
top_segment = result.iloc[0]
print(top_segment)

segment         Standard
gross_margin    0.313442
Name: 0, dtype: object


# **Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?**

> **Q3: B. wrong_size**

In [6]:
# Join returns với products
tmp = returns.merge(
    products[['product_id', 'category']],
    on='product_id',
    how='left'
)

# Lọc Streetwear
streetwear_returns = tmp[tmp['category'] == 'Streetwear']

# Đếm lý do trả hàng
result = (
    streetwear_returns['return_reason']
    .value_counts()
    .reset_index()
)

# Lý do trả hàng nhiều nhất
result.columns = ['return_reason', 'count']

reason = result.iloc[0]['return_reason']
count = result.iloc[0]['count']

print(f"Lý do trả hàng nhiều nhất: {reason} ({count} lần)")

Lý do trả hàng nhiều nhất: wrong_size (7626 lần)


# **Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?**

> **Q4: C. email_campaign**

In [7]:
# Tính bounce rate trung bình theo traffic source
result = (
    web_traffic.groupby('traffic_source')['bounce_rate']
    .mean()
    .sort_values()
    .reset_index()
)

# Cột cần thiết
result.columns = ['traffic_source', 'avg_bounce_rate']

traffic_source = result.iloc[0]['traffic_source']
avg_bounce_rate = result.iloc[0]['avg_bounce_rate']

print(f" Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: {traffic_source} ({avg_bounce_rate})")

 Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: email_campaign (0.0044584356435643565)


# **Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?**

> **Q5: C. 39%**

In [8]:
# Tỷ lệ dòng có promo_id không null
promo_rate = (
    order_items['promo_id']
    .notna()
    .mean()
) * 100

print(promo_rate)

38.663493169565214


# **Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)**

> **Q6: A. 55+**

In [9]:
# Chỉ lấy khách có age_group
cust = customers[customers['age_group'].notna()]

# Đếm số đơn của từng customer
orders_per_customer = orders.groupby('customer_id')['order_id'].count().reset_index()
orders_per_customer.rename(columns={'order_id': 'num_orders'}, inplace=True)

# Merge vào customer
tmp = cust.merge(orders_per_customer, on='customer_id', how='left')

# Khách chưa mua đơn nào => NaN -> 0
tmp['num_orders'] = tmp['num_orders'].fillna(0)

# Tính trung bình số đơn theo age_group
result = tmp.groupby('age_group')['num_orders'].mean().reset_index()

# Sắp xếp giảm dần
result = result.sort_values('num_orders', ascending=False)

age_group_result = result.iloc[0]['age_group']
num_orders_result = result.iloc[0]['num_orders']

print(f"Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất: {age_group_result} ({num_orders_result})")

Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất: 55+ (5.406851452775507)


# **Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?**

> **Q7: C. East**

In [10]:
# Merge dữ liệu
tmp = orders.merge(
    payments[['order_id', 'payment_value']],
    on='order_id',
    how='left'
).merge(
    geography[['zip', 'region']],
    on='zip',
    how='left'
)

result = (
    tmp.groupby('region')['payment_value']
    .sum()
    .reset_index(name='total_revenue')
    .sort_values('total_revenue', ascending=False)
)

result_region = result.iloc[0]['region']
result_total_revenue = result.iloc[0]['total_revenue']

print(f"Vùng tạo ra tổng doanh thu cao nhất: {result_region} ({result_total_revenue})")

Vùng tạo ra tổng doanh thu cao nhất: East (7291150819.12)


# **Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?**

> **Q8: A. credit_card**

In [11]:
# Lọc order đã cancelled
result = (
    orders[orders['order_status'] == 'cancelled']
    ['payment_method']
    .value_counts()
)

print(result)
print(f"Trong các đơn hàng cancelled, phương thức thanh toán được sử dụng nhiều nhất: {result.idxmax()} ({result.max()})")

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64
Trong các đơn hàng cancelled, phương thức thanh toán được sử dụng nhiều nhất: credit_card (28452)


# **Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?**

> **Q9: A. S**

In [12]:
# Merge dữ liệu
oi = order_items.merge(
    products[['product_id', 'size']],
    on='product_id',
    how='left'
)

rt = returns.merge(
    products[['product_id', 'size']],
    on='product_id',
    how='left'
)

# mẫu số: số dòng order_items theo size
denom = oi.groupby('size').size().reset_index(name='order_lines')

# tử số: số bản ghi returns theo size
num = rt.groupby('size').size().reset_index(name='return_records')

# ghép lại
result = denom.merge(num, on='size', how='left')

# tỷ lệ trả hàng
result['return_rate'] = result['return_records'] / result['order_lines']

# xếp hạng
result = result.sort_values('return_rate', ascending=False)

print(result)

size_result = result.iloc[0]['size']
return_rate_result = result.iloc[0]['return_rate']
print(f"Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước có tỷ lệ trả hàng cao nhất: {size_result} ({return_rate_result}) ")

  size  order_lines  return_records  return_rate
2    S       172042            9723     0.056515
0    L       173174            9741     0.056250
1    M       176428            9820     0.055660
3   XL       193025           10655     0.055200
Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước có tỷ lệ trả hàng cao nhất: S (0.05651526952720847) 


# **Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?**

> **Q10: C. 6 kỳ**

In [13]:
# Tính kế hoạch có giá trị thanh toán trung bình mỗi đơn hàng
result = (
    payments.groupby('installments')['payment_value']
    .mean()
    .reset_index(name='avg_payment')
    .sort_values('avg_payment', ascending=False)
)

print(result)

avg_payment_result = result.iloc[0]['avg_payment']
installment_result = result.iloc[0]['installments']

print(f"kế hoạch trả góp có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất: {installment_result} ({avg_payment_result})")

   installments   avg_payment
3             6  24446.654403
2             3  24399.635486
4            12  24245.772694
0             1  24113.274166
1             2    708.473729
kế hoạch trả góp có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất: 6.0 (24446.65440296606)
